# 🧠 InvestAI — Módulo 3: Clasificadores RNN (SimpleRNN · GRU · LSTM)
**Entrada:** MongoDB `precios_ohlcv` · **Salida:** `predicciones_rnn`, `metricas_modelos`
**Arquitecturas:** SimpleRNN, GRU, LSTM — clasificación binaria BUY/SELL
> ⚠️ Activar GPU: Entorno de ejecución → Cambiar tipo → T4 GPU. Requiere Secret `MONGO_URI`.

In [ ]:
import requests

# Obtener la IP pública actual de la sesión de Colab
response = requests.get('https://api.ipify.org?format=json')
public_ip = response.json()['ip']
print(f"Tu dirección IP pública actual de Colab es: {public_ip}")

Tu dirección IP pública actual de Colab es: 34.81.114.169


## Sección 1 — Instalación e Importaciones

In [ ]:
!pip install "pymongo[srv]" --quiet
import tensorflow as tf
print(f"✅ TensorFlow {tf.__version__}")
print(f"   GPU disponible: {tf.config.list_physical_devices('GPU')}")

In [ ]:
import warnings, math
warnings.filterwarnings('ignore')
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, GRU, LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

from pymongo import MongoClient
from pymongo.errors import ConnectionFailure
from google.colab import userdata

tf.random.set_seed(42)
np.random.seed(42)
print("✅ Importaciones OK.")

## Sección 2 — Configuración

In [ ]:
TICKERS = ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']
EMPRESAS_META = {
    'FSM':         {'nombre': 'Fortuna Silver Mines Inc.',             'moneda': 'USD'},
    'VOLCABC1.LM': {'nombre': 'Volcan Compania Minera S.A.A.',         'moneda': 'PEN'},
    'ABX.TO':      {'nombre': 'Barrick Gold Corporation',              'moneda': 'CAD'},
    'BVN':         {'nombre': 'Compania de Minas Buenaventura S.A.A.', 'moneda': 'USD'},
    'BHP':         {'nombre': 'BHP Group Limited',                     'moneda': 'USD'},
}

VENTANA       = 20    # Días de historia por muestra de entrada
TEST_SPLIT    = 0.20
EPOCHS        = 80
BATCH_SIZE    = 32
LEARNING_RATE = 0.001
PATIENCE_ES   = 12
PATIENCE_LR   = 6
MIN_REGISTROS = 80

ARQUITECTURAS = ['SimpleRNN', 'GRU', 'LSTM']

DB_NOMBRE        = 'investai'
COL_PRECIOS      = 'precios_ohlcv'
COL_PRED_RNN     = 'predicciones_rnn'
COL_METRICAS     = 'metricas_modelos'

COLS_FEATURES = [
    'sma_20', 'sma_50', 'ema_12', 'ema_26', 'rsi_14',
    'retorno_1d', 'retorno_3d', 'rango_intradia',
    'precio_vs_sma20', 'precio_vs_sma50', 'cruce_ema',
]

print(f"Ventana: {VENTANA} días  |  Arquitecturas: {ARQUITECTURAS}")
print("✅ Configuración OK.")

## Sección 3 — Conexión a MongoDB

In [ ]:
try:
    MONGO_URI = userdata.get('MONGO_URI')
    if not MONGO_URI:
        raise ValueError("Secret MONGO_URI vacío.")
except Exception as e:
    raise RuntimeError(f"No se encontró MONGO_URI: {e}")

try:
    cliente = MongoClient(MONGO_URI, serverSelectionTimeoutMS=10_000)
    cliente.admin.command('ping')
    print("✅ Conexión a MongoDB Atlas establecida.")
except ConnectionFailure as e:
    raise RuntimeError(f"Falla de conexión: {e}")

db           = cliente[DB_NOMBRE]
col_precios  = db[COL_PRECIOS]
col_pred_rnn = db[COL_PRED_RNN]
col_metricas = db[COL_METRICAS]

col_pred_rnn.create_index(
    [('ticker', 1), ('arquitectura', 1)],
    unique=True, name='idx_ticker_arq'
)

n = col_precios.count_documents({})
print(f"   precios_ohlcv: {n:,} documentos")
if n == 0:
    print("⚠️  Ejecuta primero el Módulo 1 (Ingesta de Datos).")

## Sección 4 — Funciones de Utilidad

In [ ]:
def nan_a_none(v):
    if v is None: return None
    try:
        return None if (math.isnan(float(v)) or math.isinf(float(v))) else round(float(v), 6)
    except (TypeError, ValueError):
        return None


def leer_precios(ticker):
    docs = list(col_precios.find({'ticker': ticker}, {'_id': 0}).sort('fecha', 1))
    if not docs:
        return None
    df = pd.DataFrame(docs)
    df['fecha'] = pd.to_datetime(df['fecha'])
    df = df.set_index('fecha').sort_index()
    return df[~df.index.duplicated(keep='last')]


def construir_features(df):
    """Añade features derivadas y target binario sin leakage futuro."""
    d = df.copy()
    d['retorno_1d']      = d['cierre'].pct_change(1)
    d['retorno_3d']      = d['cierre'].pct_change(3)
    d['rango_intradia']  = (d['maximo'] - d['minimo']) / d['cierre']
    d['precio_vs_sma20'] = d['cierre'] / d['sma_20'] - 1
    d['precio_vs_sma50'] = d['cierre'] / d['sma_50'] - 1
    d['cruce_ema']       = d['ema_12'] / d['ema_26'] - 1
    d['target'] = (d['cierre'].shift(-1) > d['cierre']).astype(int)
    return d


def construir_modelo(arq, ventana, n_feat):
    """Construye y compila modelo de clasificación binaria según arquitectura."""
    modelo = Sequential(name=f'{arq}_Clasificador')
    modelo.add(Input(shape=(ventana, n_feat)))

    if arq == 'SimpleRNN':
        modelo.add(SimpleRNN(64))
        modelo.add(Dropout(0.2))
    elif arq == 'GRU':
        modelo.add(GRU(64, return_sequences=True))
        modelo.add(Dropout(0.2))
        modelo.add(GRU(32))
        modelo.add(Dropout(0.2))
    elif arq == 'LSTM':
        modelo.add(LSTM(64, return_sequences=True))
        modelo.add(Dropout(0.2))
        modelo.add(LSTM(32))
        modelo.add(Dropout(0.2))

    modelo.add(Dense(16, activation='relu'))
    modelo.add(Dense(1,  activation='sigmoid'))
    modelo.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return modelo

print("✅ Funciones de utilidad definidas.")

## Sección 5 — Pipeline de Entrenamiento

In [ ]:
def entrenar_rnn(ticker, arq):
    # 1. Leer y validar
    df_raw = leer_precios(ticker)
    if df_raw is None or len(df_raw) < MIN_REGISTROS + VENTANA:
        print(f"   ⚠️  {ticker}/{arq}: datos insuficientes.")
        return None

    # 2. Features + target
    df_feat = construir_features(df_raw)
    df_feat = df_feat.dropna(subset=COLS_FEATURES + ['target']).copy()
    if len(df_feat) < MIN_REGISTROS + VENTANA:
        print(f"   ⚠️  {ticker}/{arq}: solo {len(df_feat)} filas válidas.")
        return None

    fila_vig = df_feat.iloc[[-1]]
    df_model = df_feat.iloc[:-1]   # Excluir última fila (target desconocido hoy)

    X_all = df_model[COLS_FEATURES].values
    y_all = df_model['target'].values

    # 3. Normalizar (scaler sobre todos para estabilidad en series cortas)
    scaler    = MinMaxScaler()
    X_all_n   = scaler.fit_transform(X_all)

    # 4. Ventanas deslizantes
    Xv, yv = [], []
    for i in range(VENTANA, len(X_all_n)):
        Xv.append(X_all_n[i - VENTANA:i])
        yv.append(y_all[i])
    Xv = np.array(Xv)
    yv = np.array(yv)

    # 5. Partición temporal (sin shuffle)
    corte = int(len(Xv) * (1 - TEST_SPLIT))
    X_train, X_test = Xv[:corte], Xv[corte:]
    y_train, y_test = yv[:corte], yv[corte:]

    if len(X_test) < 5:
        print(f"   ⚠️  {ticker}/{arq}: test demasiado pequeño.")
        return None

    # 6. Modelo + entrenamiento
    modelo = construir_modelo(arq, VENTANA, len(COLS_FEATURES))
    cbs = [
        EarlyStopping(monitor='val_loss', patience=PATIENCE_ES,
                      restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', patience=PATIENCE_LR,
                          factor=0.5, min_lr=1e-6, verbose=0),
    ]
    hist = modelo.fit(
        X_train, y_train,
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        validation_split=0.1, callbacks=cbs, verbose=0
    )

    # 7. Evaluación
    y_prob = modelo.predict(X_test, verbose=0).flatten()
    y_pred = (y_prob >= 0.5).astype(int)
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

    metricas = {
        'accuracy':  nan_a_none(accuracy_score(y_test, y_pred)),
        'precision': nan_a_none(precision_score(y_test, y_pred, zero_division=0)),
        'recall':    nan_a_none(recall_score(y_test, y_pred, zero_division=0)),
        'f1':        nan_a_none(f1_score(y_test, y_pred, zero_division=0)),
    }

    # 8. Señal vigente
    X_vig_n = scaler.transform(fila_vig[COLS_FEATURES].values)
    X_vig_seq = X_all_n[-VENTANA:].reshape(1, VENTANA, len(COLS_FEATURES))
    prob_vig  = float(modelo.predict(X_vig_seq, verbose=0).flatten()[0])
    pred_vig  = 1 if prob_vig >= 0.5 else 0
    senal     = 'BUY' if pred_vig == 1 else 'SELL'
    confianza = prob_vig if pred_vig == 1 else 1 - prob_vig

    return {
        'ticker': ticker, 'arquitectura': arq,
        'n_train': len(X_train), 'n_test': len(X_test),
        'epocas_reales': len(hist.history['loss']),
        'metricas': metricas,
        'matriz_confusion': cm.tolist(),
        'senal_vigente': senal,
        'confianza': nan_a_none(confianza),
        'ultimo_cierre': nan_a_none(float(df_feat['cierre'].iloc[-1])),
        'fecha_vigente': fila_vig.index[0].strftime('%Y-%m-%d'),
    }

print("✅ Función entrenar_rnn() definida.")

## Sección 6 — Persistencia y Pipeline Principal

In [ ]:
def guardar_pred_rnn(res):
    meta = EMPRESAS_META.get(res['ticker'], {})
    doc = {
        'ticker': res['ticker'], 'nombre': meta.get('nombre', res['ticker']),
        'modelo': 'RNN', 'arquitectura': res['arquitectura'],
        'senal': res['senal_vigente'], 'confianza': res['confianza'],
        'ultimo_cierre': res['ultimo_cierre'],
        'fecha_referencia': res['fecha_vigente'],
        'epocas_reales': res['epocas_reales'],
        'actualizado_en': datetime.now(timezone.utc),
    }
    col_pred_rnn.update_one(
        {'ticker': res['ticker'], 'arquitectura': res['arquitectura']},
        {'$set': doc}, upsert=True
    )

def guardar_met_rnn(res):
    meta = EMPRESAS_META.get(res['ticker'], {})
    col_metricas.insert_one({
        'ticker': res['ticker'], 'nombre': meta.get('nombre', res['ticker']),
        'modelo': f"RNN_{res['arquitectura']}", 'arquitectura': res['arquitectura'],
        'n_train': res['n_train'], 'n_test': res['n_test'],
        'ventana_dias': VENTANA, 'epocas_reales': res['epocas_reales'],
        'metricas': res['metricas'], 'matriz_confusion': res['matriz_confusion'],
        'entrenado_en': datetime.now(timezone.utc),
    })

print("✅ Funciones de persistencia definidas.")

In [ ]:
resumen = []
print("=" * 65)
print("  CLASIFICADORES RNN — InvestAI (SimpleRNN · GRU · LSTM)")
print("=" * 65)

for ticker in TICKERS:
    meta = EMPRESAS_META.get(ticker, {})
    print(f"\n{'='*55}")
    print(f"  🔲 {ticker}  ({meta.get('nombre','')})")
    print(f"{'='*55}")

    for arq in ARQUITECTURAS:
        print(f"\n  🧠 {arq}")
        res = entrenar_rnn(ticker, arq)
        if res is None:
            resumen.append({'ticker': ticker, 'arq': arq, 'estado': '❌'})
            continue

        m = res['metricas']
        print(f"     Épocas: {res['epocas_reales']}  "
              f"Train/Test: {res['n_train']}/{res['n_test']}")
        print(f"     Acc: {m['accuracy']}  F1: {m['f1']}")
        print(f"     📌 Señal: {res['senal_vigente']}  "
              f"(confianza: {res['confianza']:.2%})")

        guardar_pred_rnn(res)
        guardar_met_rnn(res)
        print("     ✅ Guardado en MongoDB.")

        resumen.append({'ticker': ticker, 'arq': arq, 'estado': '✅',
                        'senal': res['senal_vigente'], 'f1': m['f1']})

print("\n" + "=" * 65)
print(pd.DataFrame(resumen).to_string(index=False))
print("=" * 65)

## Sección 7 — Verificación Final

In [ ]:
n_pred = col_pred_rnn.count_documents({})
print(f"predicciones_rnn: {n_pred} docs "
      f"(esperado: {len(TICKERS) * len(ARQUITECTURAS)})")

for doc in col_pred_rnn.find({}, {'_id': 0}).sort([('ticker',1),('arquitectura',1)]):
    print(f"  {doc['ticker']:<14} {doc['arquitectura']:<12} "
          f"señal: {doc['senal']:<5}  conf: {doc.get('confianza',0):.2%}")

invalidas = col_pred_rnn.count_documents({'senal': {'$nin': ['BUY','SELL']}})
print(f"\n✅ Señales inválidas: {invalidas}  (debe ser 0)")
cliente.close()
print("✅ Verificación OK. Conexión cerrada.")